# 🔍 Policy-Aware Code Compliance Auditor
### RAG + Fine-Tuned CodeBERT Hybrid Pipeline
---

In [1]:
!pip install -q faiss-cpu sentence-transformers google-generativeai requests
!pip install -q google-genai
!pip install -q transformers==4.41.2 peft==0.11.1 accelerate==0.30.1
!pip install -q reportlab


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 90.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 88.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.6/302.6 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 112.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 39.9 MB/s eta 0:00:00


## Step 1 — API Keys

In [2]:
from google.colab import userdata
import os

os.environ["GROK_API_KEY"]   = userdata.get("GROK_API_KEY")
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
os.environ["GROQ_API_KEY"]   = os.environ["GROK_API_KEY"]

# Optional — only needed if loading classifier from HuggingFace Hub
# os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

print("✅ API keys loaded")


✅ API keys loaded


## Step 2 — Policy Definitions

In [3]:
policies = [
  # 💰 Payments
  "All payment transactions must be logged before and after execution using AuditLogger.",
  "Only approved payment providers may be used via PaymentService.",
  "No direct database updates to payment or order status are allowed.",

  # 🎯 Discounts / Offers
  "Discounts must not be hardcoded and must be configurable via the offer system.",
  "All discount applications must go through the OfferService or equivalent module.",

  # 👤 Customer Data (GDPR-style)
  "Customer PII must only be accessed via Customer service classes.",
  "Customer email and personal data must not be directly queried from the database.",

  # 📢 Communication
  "All user communications must go through a centralized communication service.",
  "User opt-out preferences must always be respected before sending emails.",

  # ⚙️ Architecture / Workflow
  "Business logic must not bypass service layers and directly manipulate database models.",
  "All order state transitions must follow the defined workflow system.",

  # 🔍 Logging & Observability
  "Critical operations must include structured logging for auditing purposes.",
]

print(f"✅ {len(policies)} policies loaded")


✅ 12 policies loaded


## Step 3 — Gemini Embeddings + FAISS Index

In [4]:
import faiss
import numpy as np
from google import genai
from google.genai import types

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
EMBED_MODEL = "models/gemini-embedding-2-preview"

def embed_texts(texts: list[str]) -> np.ndarray:
    embeddings = []
    for text in texts:
        response = client.models.embed_content(
            model=EMBED_MODEL,
            contents=text,
            config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT")
        )
        embeddings.append(response.embeddings[0].values)
    return np.array(embeddings, dtype="float32")

def embed_query(text: str) -> np.ndarray:
    response = client.models.embed_content(
        model=EMBED_MODEL,
        contents=text,
        config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY")
    )
    return np.array(response.embeddings[0].values, dtype="float32").reshape(1, -1)

print("⏳ Embedding policies...")
policy_embeddings = embed_texts(policies)

dimension = policy_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(policy_embeddings)

print(f"✅ FAISS index built — {index.ntotal} policies, dimension {dimension}")


⏳ Embedding policies...
✅ FAISS index built — 12 policies, dimension 3072


## Step 4 — Load Fine-Tuned CodeBERT Classifier
> The model was saved from `jogi.ipynb` (Section 11) to `/content/policy_classifier`.
> Mount Google Drive or upload the folder, then set `CLASSIFIER_PATH` below.
>
> **If you don't have the saved model yet**, run `jogi.ipynb` first to train and save it,
> then come back here. The pipeline will fall back to LLM-only mode if the path is wrong.


In [5]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# ── CONFIGURE THIS PATH ────────────────────────────────────────────────────
# Option A: Model saved locally in Colab from jogi.ipynb
# CLASSIFIER_PATH = "/content/policy_classifier"

# Option B: Model on Google Drive — uncomment and mount Drive first
from google.colab import drive
drive.mount('/content/drive')
CLASSIFIER_PATH = "/content/drive/MyDrive/policy_classifier"
# ──────────────────────────────────────────────────────────────────────────

clf_model = None
clf_tok   = None

import os as _os
if _os.path.isdir(CLASSIFIER_PATH):
    try:
        clf_tok   = AutoTokenizer.from_pretrained(CLASSIFIER_PATH)
        clf_model = AutoModelForSequenceClassification.from_pretrained(CLASSIFIER_PATH)
        clf_model.to(device).eval()
        print(f"✅ Fine-tuned classifier loaded from {CLASSIFIER_PATH}")
        print(f"   Labels: {clf_model.config.id2label}")
    except Exception as e:
        print(f"⚠️  Could not load classifier: {e}")
        print("   Pipeline will run in LLM-only mode.")
else:
    print(f"⚠️  Classifier path not found: {CLASSIFIER_PATH}")
    print("   Run jogi.ipynb first to train and save the model.")
    print("   Pipeline will run in LLM-only mode.")


Device: cuda
Mounted at /content/drive


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at microsoft/codebert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Fine-tuned classifier loaded from /content/drive/MyDrive/policy_classifier
   Labels: {0: 'LABEL_0', 1: 'LABEL_1'}


## Step 5 — Code Chunking (AST)

In [6]:
import ast
import textwrap

def parse_code_chunks(source_code: str, filename="code.py") -> list[dict]:
    """Extract functions and classes as individual chunks using Python's AST."""
    chunks = []
    try:
        tree = ast.parse(source_code)
    except SyntaxError:
        return [{"name": filename, "code": source_code}]

    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            chunk_code = ast.get_source_segment(source_code, node)
            if chunk_code:
                chunks.append({
                    "name": node.name,
                    "type": type(node).__name__,
                    "code": textwrap.dedent(chunk_code)
                })
    return chunks if chunks else [{"name": filename, "code": source_code}]


## Step 6 — Groq API Helpers (Summarizer + LLM Auditor)

In [7]:
import requests
import json
import time

GROQ_API_URL         = "https://api.groq.com/openai/v1/chat/completions"
GROQ_SUMMARIZE_MODEL = "llama-3.1-8b-instant"
GROQ_AUDIT_MODEL     = "moonshotai/kimi-k2-instruct"

def call_groq_with_retry(payload: dict, max_retries: int = 5) -> dict:
    """Call Groq API with automatic retry on rate-limit."""
    headers = {
        "Authorization": f"Bearer {os.environ['GROQ_API_KEY']}",
        "Content-Type": "application/json"
    }
    for attempt in range(max_retries):
        response = requests.post(GROQ_API_URL, headers=headers, json=payload)
        data = response.json()
        if "choices" in data:
            return data
        error = data.get("error", {})
        if error.get("code") == "rate_limit_exceeded":
            msg = error.get("message", "")
            wait = 10.0
            try:
                wait = float(msg.split("in ")[-1].replace("s.","").replace("s","").strip()) + 1.0
            except:
                pass
            print(f"⏳ Rate limit — waiting {wait:.1f}s (attempt {attempt+1}/{max_retries})")
            time.sleep(wait)
        else:
            return data
    print("❌ Max retries exceeded")
    return {}

def summarize_with_grok(code_chunk: str) -> str:
    """Summarize a code chunk in 2-3 sentences (business logic focus)."""
    payload = {
        "model": GROQ_SUMMARIZE_MODEL,
        "messages": [
            {"role": "system", "content": "You are a code analyst. Describe what this code does in 2-3 sentences, focusing on business logic, data access, and external calls."},
            {"role": "user",   "content": f"```python\n{code_chunk}\n```"}
        ],
        "max_tokens": 150
    }
    data = call_groq_with_retry(payload)
    if "choices" not in data:
        return f"Summary unavailable: {data.get('error', {}).get('message', str(data))}"
    return data["choices"][0]["message"]["content"].strip()

def llm_audit_chunk(
    chunk_name: str,
    code: str,
    summary: str,
    relevant_policies: list[str],
    classifier_hint: str = "",
) -> dict:
    """
    Full LLM audit via Kimi K2.  When called from the hybrid pipeline,
    classifier_hint carries the pre-classification result so the LLM
    can use it as additional context and produce a richer explanation.
    """
    policy_text = "\n".join(f"- {p}" for p in relevant_policies)
    hint_block  = (
        f"\nPRE-CLASSIFICATION HINT (fine-tuned CodeBERT): {classifier_hint}\n"
        if classifier_hint else ""
    )
    prompt = (
        f"You are a compliance auditor. Analyze the following code against the provided policies.\n\n"
        f"CODE CHUNK: {chunk_name}\n```python\n{code}\n```\n\n"
        f"CODE SUMMARY: {summary}\n"
        f"{hint_block}\n"
        f"RELEVANT POLICIES:\n{policy_text}\n\n"
        "Respond in JSON with this exact format:\n"
        '{"compliant": true or false, '
        '"violations": ["list of specific violations found, empty if none"], '
        '"explanation": "clear explanation of your reasoning — state which policy is violated and why, '
        'or why the code is compliant", '
        '"severity": "none | low | medium | high | critical"}'
    )
    payload = {
        "model": GROQ_AUDIT_MODEL,
        "messages": [
            {"role": "system", "content": "You are a policy compliance auditor. Always respond with valid JSON only. No markdown, no text outside the JSON object."},
            {"role": "user",   "content": prompt}
        ],
        "max_tokens": 500
    }
    data = call_groq_with_retry(payload)
    if "choices" not in data:
        return {"compliant": False, "violations": ["API error"], "explanation": str(data), "severity": "unknown"}
    raw   = data["choices"][0]["message"]["content"].strip()
    clean = raw.replace("```json","").replace("```","").strip()
    try:
        return json.loads(clean)
    except json.JSONDecodeError:
        return {"compliant": False, "violations": ["JSON parse error"], "explanation": raw, "severity": "unknown"}

print("✅ Groq helpers loaded")


✅ Groq helpers loaded


## Step 7 — Policy Retrieval (FAISS)

In [8]:
def retrieve_relevant_policies(code_summary: str, top_k: int = 3) -> list[str]:
    """Embed the code summary and find the most relevant policies."""
    query_embedding = embed_query(code_summary)
    distances, indices = index.search(query_embedding, top_k)
    return [policies[i] for i in indices[0] if i < len(policies)]


## Step 8 — Hybrid Audit Engine
> **Decision logic for each (code chunk, policy) pair:**
>
> | Classifier confidence | Action |
> |---|---|
> | > 65% (high confidence violation) | Classifier result used directly — LLM writes explanation |
> | < 35% (high confidence compliant) | Classifier result used directly — LLM writes explanation |
> | 35–65% (ambiguous) | Full LLM audit with classifier hint |
>
> Both compliant and non-compliant chunks always receive a full natural-language explanation.


In [9]:
# ── Constants ───────────────────────────────────────────────────────────────
AMBIGUITY_LOW  = 0.35   # below this  → high-confidence COMPLIANT
AMBIGUITY_HIGH = 0.65   # above this  → high-confidence VIOLATION
CLF_MAX_LENGTH = 512

_SEVERITY_RANK = {"none": 0, "low": 1, "medium": 2, "high": 3, "critical": 4, "unknown": 0}


def _classify_single(code: str, policy: str) -> dict:
    """
    Run the fine-tuned CodeBERT classifier on one (code, policy) pair.
    Returns a dict with keys: compliant, violation_probability, severity.
    Returns None if classifier is not loaded.
    """
    if clf_model is None or clf_tok is None:
        return None

    enc = clf_tok(
        code, policy,
        max_length=CLF_MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    )
    enc = {k: v.to(device) for k, v in enc.items()}

    with torch.no_grad():
        probs = torch.softmax(clf_model(**enc).logits, dim=-1)[0]

    vp = probs[1].item()   # probability of VIOLATION

    if vp >= AMBIGUITY_HIGH:
        sev = "critical" if vp >= 0.92 else ("high" if vp >= 0.80 else "medium")
    elif vp >= AMBIGUITY_LOW:
        sev = "low"    # ambiguous — will escalate to LLM
    else:
        sev = "none"

    return {
        "compliant":             vp < AMBIGUITY_HIGH,
        "violation_probability": round(vp, 4),
        "severity":              sev,
    }


def _llm_explain(
    chunk_name: str,
    code: str,
    summary: str,
    relevant_policies: list[str],
    clf_result: dict | None,
    force_verdict: bool | None = None,
) -> dict:
    """
    Always calls the LLM.  If force_verdict is set (True=compliant, False=violation),
    the classifier's decision is trusted and the LLM is asked only for the explanation.
    If force_verdict is None, the LLM makes its own decision (ambiguous cases).
    """
    if clf_result is not None:
        vp   = clf_result["violation_probability"]
        hint = (
            f"Fine-tuned CodeBERT: p(violation)={vp:.1%}, "
            f"preliminary verdict={'VIOLATION' if not clf_result['compliant'] else 'COMPLIANT'}. "
            + ("Please confirm and explain in detail." if force_verdict is None
               else "The classifier is confident — provide a detailed explanation consistent with this verdict.")
        )
    else:
        hint = ""

    return llm_audit_chunk(chunk_name, code, summary, relevant_policies, classifier_hint=hint)


def audit_chunk_hybrid(
    chunk_name: str,
    code: str,
    summary: str,
    relevant_policies: list[str],
) -> dict:
    """
    Hybrid compliance audit for one code chunk against its retrieved policies.

    - Runs CodeBERT classifier per policy for an initial signal.
    - High-confidence results (> 65% or < 35%) skip the full LLM reasoning call
      but still request a natural-language explanation from the LLM.
    - Ambiguous results (35–65%) trigger a full LLM audit with the classifier hint.
    - If classifier is unavailable, falls back to LLM-only.

    Returns the same dict schema as the original audit_chunk_with_grok().
    """
    # ── Fallback: no classifier ──────────────────────────────────────────────
    if clf_model is None:
        return _llm_explain(chunk_name, code, summary, relevant_policies, None)

    # ── Run classifier per policy ────────────────────────────────────────────
    clf_results   = []
    any_violation = False
    any_ambiguous = False
    max_severity  = "none"

    for policy in relevant_policies:
        r = _classify_single(code, policy)
        if r is None:
            break
        clf_results.append((policy, r))
        vp = r["violation_probability"]
        if vp >= AMBIGUITY_HIGH:
            any_violation = True
        elif AMBIGUITY_LOW <= vp <= AMBIGUITY_HIGH:
            any_ambiguous = True
        if _SEVERITY_RANK.get(r["severity"], 0) > _SEVERITY_RANK.get(max_severity, 0):
            max_severity = r["severity"]

    # Summarise classifier signal for the LLM hint
    if clf_results:
        best_vp  = max(r["violation_probability"] for _, r in clf_results)
        clf_summary = {
            "compliant":             not any_violation,
            "violation_probability": round(best_vp, 4),
            "severity":              max_severity,
        }
    else:
        clf_summary = None

    # ── Always get LLM explanation (with different depths) ───────────────────
    if clf_summary is None:
        # Classifier failed mid-way — go pure LLM
        llm = _llm_explain(chunk_name, code, summary, relevant_policies, None)
    elif any_ambiguous or (not any_violation and not any_ambiguous is False):
        # Ambiguous case: full LLM reasoning
        llm = _llm_explain(chunk_name, code, summary, relevant_policies, clf_summary, force_verdict=None)
    else:
        # High-confidence: LLM explains the classifier's finding
        llm = _llm_explain(chunk_name, code, summary, relevant_policies, clf_summary,
                           force_verdict=(not any_violation))

    # ── Merge: trust LLM verdict, enrich with classifier confidence ──────────
    explanation = llm.get("explanation", "")
    if clf_summary is not None:
        mode = "ambiguous→full-LLM" if any_ambiguous else "high-confidence→LLM-explain"
        explanation = (
            f"[Hybrid/{mode}] "
            f"Classifier p(violation)={clf_summary['violation_probability']:.1%}. "
            + explanation
        )

    return {
        "compliant":   llm.get("compliant", True),
        "violations":  llm.get("violations", []),
        "explanation": explanation,
        "severity":    llm.get("severity", "none"),
    }


print("✅ Hybrid audit engine loaded")
print(f"   Classifier available: {clf_model is not None}")
print(f"   Mode: {'Hybrid (classifier + LLM explanation)' if clf_model else 'LLM-only (fallback)'}")


✅ Hybrid audit engine loaded
   Classifier available: True
   Mode: Hybrid (classifier + LLM explanation)


## Step 9 — Compliance Audit Pipeline

In [10]:
def run_compliance_audit(source_code: str, filename: str = "code.py") -> dict:
    """
    Full compliance audit pipeline:
      1. AST-chunk the source code
      2. Summarize each chunk with LLaMA 8B
      3. Retrieve top-3 relevant policies via FAISS
      4. Hybrid audit: CodeBERT classifier + Kimi K2 explanation
      5. Return structured report
    """
    print(f"\n🔍 Auditing: {filename}")
    chunks = parse_code_chunks(source_code, filename)
    print(f"   Found {len(chunks)} code chunks")

    report = {
        "file":    filename,
        "chunks":  [],
        "summary": {"total": 0, "violations": 0, "critical": 0}
    }

    for chunk in chunks:
        print(f"   → Analyzing: {chunk['name']}")

        summary           = summarize_with_grok(chunk["code"])
        relevant_policies = retrieve_relevant_policies(summary)
        audit             = audit_chunk_hybrid(
            chunk["name"], chunk["code"], summary, relevant_policies
        )

        report["chunks"].append({
            "name":             chunk["name"],
            "summary":          summary,
            "policies_checked": relevant_policies,
            "compliant":        audit.get("compliant", False),
            "violations":       audit.get("violations", []),
            "explanation":      audit.get("explanation", ""),
            "severity":         audit.get("severity", "none"),
        })

        report["summary"]["total"] += 1
        if not audit.get("compliant", True):
            report["summary"]["violations"] += 1
        if audit.get("severity") == "critical":
            report["summary"]["critical"] += 1

        # Reduced sleep — classifier handles most chunks locally
        time.sleep(1)

    return report


def print_report(report: dict):
    """Pretty-print a compliance report to stdout."""
    print("\n" + "="*60)
    print(f"📋 COMPLIANCE REPORT — {report['file']}")
    print("="*60)
    s = report["summary"]
    print(f"   Chunks analyzed : {s['total']}")
    print(f"   Violations found: {s['violations']}")
    print(f"   Critical issues : {s['critical']}")
    print()

    for chunk in report["chunks"]:
        if chunk["compliant"]:
            print(f"  ✅ COMPLIANT — {chunk['name']}")
        else:
            sev = chunk['severity'].upper()
            print(f"  ❌ VIOLATION [{sev}] — {chunk['name']}")

        print(f"    Summary  : {chunk['summary'][:120]}...")

        # Always print explanation (both compliant and non-compliant)
        print(f"    Reason   : {chunk['explanation'][:250]}...")

        if chunk["violations"]:
            print(f"    Issues:")
            for v in chunk["violations"]:
                print(f"      ⚠️  {v}")
        else:
            print(f"    ✔ No specific policy violations detected.")
        print()


print("✅ run_compliance_audit + print_report loaded")


✅ run_compliance_audit + print_report loaded


## Step 10 — Clone Repository and Run Audit

In [11]:
import shutil

# ── CONFIG ──────────────────────────────────────────────────────────────────
GITHUB_REPO = "https://github.com/rtk5/code-sample"
CLONE_PATH  = "/content/repo"
# ────────────────────────────────────────────────────────────────────────────

if os.path.exists(CLONE_PATH):
    shutil.rmtree(CLONE_PATH)

import subprocess
subprocess.run(["git", "clone", GITHUB_REPO, CLONE_PATH], check=True)

py_files = []
SKIP_DIRS = {".git", "__pycache__", ".venv", "venv", "node_modules", "migrations"}
for root, dirs, files in os.walk(CLONE_PATH):
    dirs[:] = [d for d in dirs if d not in SKIP_DIRS]
    for f in files:
        if f.endswith(".py"):
            py_files.append(os.path.join(root, f))

print(f"\n📁 Found {len(py_files)} Python files to audit")
print(f"🤖 Mode: {'Hybrid (CodeBERT + Kimi K2)' if clf_model else 'LLM-only (Kimi K2)'}\n")

all_reports      = []
total_chunks     = 0
total_violations = 0
total_critical   = 0

for path in py_files:
    with open(path, encoding="utf-8", errors="ignore") as fp:
        code = fp.read()
    if len(code.strip()) < 50:
        continue
    relative_path = path.replace(CLONE_PATH + "/", "")
    r = run_compliance_audit(code, relative_path)
    print_report(r)
    all_reports.append(r)
    total_chunks     += r["summary"]["total"]
    total_violations += r["summary"]["violations"]
    total_critical   += r["summary"]["critical"]

print("\n" + "="*60)
print("🏁  FULL REPO AUDIT COMPLETE")
print("="*60)
print(f"   Files audited    : {len(all_reports)}")
print(f"   Chunks analyzed  : {total_chunks}")
print(f"   Total violations : {total_violations}")
print(f"   Critical issues  : {total_critical}")
compliance_rate = round((total_chunks - total_violations) / total_chunks * 100) if total_chunks else 0
print(f"   Compliance rate  : {compliance_rate}%")
print("="*60)



📁 Found 9 Python files to audit
🤖 Mode: Hybrid (CodeBERT + Kimi K2)


🔍 Auditing: payment/utils.py
   Found 19 code chunks
   → Analyzing: OrderNumberGenerator
   → Analyzing: OrderCreator
   → Analyzing: OrderDispatcher
   → Analyzing: order_number
   → Analyzing: place_order
   → Analyzing: create_order_model
   → Analyzing: create_line_models
   → Analyzing: update_stock_records
   → Analyzing: create_line_discount_models
   → Analyzing: create_additional_line_models
   → Analyzing: create_line_price_models
   → Analyzing: create_line_attributes
   → Analyzing: create_discount_model
   → Analyzing: record_discount
   → Analyzing: record_voucher_usage
   → Analyzing: __init__
   → Analyzing: dispatch_order_messages
   → Analyzing: create_communication_event
   → Analyzing: send_order_placed_email_for_user

📋 COMPLIANCE REPORT — payment/utils.py
   Chunks analyzed : 19
   Violations found: 1
   Critical issues : 0

  ✅ COMPLIANT — OrderNumberGenerator
    Summary  : This code defines

## Step 11 — Save Reports (JSON + PDF)

In [14]:
import json
import html
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, HRFlowable
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors

# ── SAFE TEXT HELPER ──────────────────────────────────────────────────────────
def safe_text(text):
    if not isinstance(text, str):
        text = str(text)
    return html.escape(text)

# ── JSON SAVE ─────────────────────────────────────────────────────────────────
with open("audit_report.json", "w") as f:
    json.dump({
        "reports": all_reports,
        "summary": {
            "files":      len(all_reports),
            "chunks":     total_chunks,
            "violations": total_violations,
            "critical":   total_critical,
        },
        "audit_mode": "hybrid" if clf_model else "llm-only",
    }, f, indent=4)

print("✅ audit_report.json saved")

# ── PDF GENERATION ────────────────────────────────────────────────────────────
def generate_pdf_report(all_reports: list, filename: str = "audit_report.pdf"):
    doc = SimpleDocTemplate(filename)
    styles = getSampleStyleSheet()

    violation_style = ParagraphStyle(
        "ViolationText",
        parent=styles["Normal"],
        textColor=colors.red,
        spaceAfter=4
    )

    compliant_style = ParagraphStyle(
        "CompliantText",
        parent=styles["Normal"],
        textColor=colors.green,
        spaceAfter=4
    )

    normal_style = ParagraphStyle(
        "NormalText",
        parent=styles["Normal"],
        spaceAfter=4
    )

    content = []

    # ── TITLE ────────────────────────────────────────────────────────────────
    content.append(Paragraph("Policy-Aware Code Compliance Audit Report", styles["Title"]))
    mode_label = "Hybrid (CodeBERT + LLM)" if clf_model else "LLM-only"
    content.append(Paragraph(f"Audit Mode: {safe_text(mode_label)}", normal_style))
    content.append(Spacer(1, 12))

    # ── FILE LOOP ────────────────────────────────────────────────────────────
    for report in all_reports:
        content.append(Paragraph(f"File: {safe_text(report['file'])}", styles["Heading2"]))

        s = report["summary"]
        summary_line = f"Chunks: {s['total']} | Violations: {s['violations']} | Critical: {s['critical']}"
        content.append(Paragraph(safe_text(summary_line), normal_style))
        content.append(Spacer(1, 6))

        # ── CHUNK LOOP ───────────────────────────────────────────────────────
        for chunk in report["chunks"]:

            name = safe_text(chunk.get("name", "unknown"))

            if chunk.get("compliant", False):
                status = safe_text(f"✔ COMPLIANT — {name}")
                content.append(Paragraph(f"<b>{status}</b>", compliant_style))
            else:
                sev = safe_text(chunk.get("severity", "unknown").upper())
                status = safe_text(f"✘ VIOLATION [{sev}] — {name}")
                content.append(Paragraph(f"<b>{status}</b>", violation_style))

            # Summary
            summary_text = safe_text(chunk.get("summary", "")[:300])
            content.append(Paragraph(f"Summary: {summary_text}", normal_style))

            # Explanation
            explanation_text = safe_text(chunk.get("explanation", "")[:500])
            content.append(Paragraph(f"Reason: {explanation_text}", normal_style))

            # Violations
            violations = chunk.get("violations", [])
            if violations:
                content.append(Paragraph("Policy Violations:", normal_style))
                for v in violations:
                    content.append(Paragraph(f"⚠ {safe_text(v)}", violation_style))
            else:
                content.append(Paragraph("No specific policy violations detected.", compliant_style))

            content.append(Spacer(1, 10))

        content.append(HRFlowable(width="100%", thickness=0.5, color=colors.grey))
        content.append(Spacer(1, 12))

    doc.build(content)
    print(f"✅ PDF report saved as {filename}")


# ── RUN ───────────────────────────────────────────────────────────────────────
generate_pdf_report(all_reports)

✅ audit_report.json saved
✅ PDF report saved as audit_report.pdf


## Step 12 — Download Reports

In [15]:
from google.colab import files
files.download("audit_report.json")
files.download("audit_report.pdf")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>